# 🧹 Phase 1: Data Cleaning & Quality Validation Pipeline
**Project:** E-Commerce Sales Analytics Portfolio Project
**Objective:** Ingest raw transactions without modifying the source, parse datetimes, standardize postal code representations, validate business domain rules, handle legitimate negative profit transactions, and engineer core temporal and margin features.

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# File Paths
RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'Sample_Superstore.csv')
CLEANED_DATA_PATH = os.path.join('..', 'data', 'cleaned', 'superstore_cleaned.csv')

# 1. Load Raw Data (Without Modifying Raw File)
df_raw = pd.read_csv(RAW_DATA_PATH, encoding='windows-1252')
print(f'Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
df_raw.head(3)

## 1. Data Cleaning Transformations
1. Standardize column names to lower snake_case.
2. Convert `order_date` and `ship_date` into datetime.
3. Convert `postal_code` to 5-digit string format (padding 449 East Coast records with leading zeros).
4. Check for duplicate rows.
5. Verify missing values across all columns.

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]

# Parse dates
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')
df['ship_date'] = pd.to_datetime(df['ship_date'], format='mixed')

# Handle Postal Code as 5-digit string
df['postal_code'] = df['postal_code'].astype(str).str.zfill(5)

# Check duplicates & nulls
print(f'Exact full duplicates: {df.duplicated().sum()}')
print(f'Total null values: {df.isnull().sum().sum()}')

## 2. Feature Engineering
Deriving analytical dimensions:
- `year`, `month_number`, `month`, `quarter`, `year_month` (ensuring `year_month` sorts chronologically in `YYYY-MM` format)
- `shipping_days` (`ship_date - order_date`)
- `profit_margin` (`profit / sales`)

In [ ]:
df['year'] = df['order_date'].dt.year
df['month_number'] = df['order_date'].dt.month
df['month'] = df['order_date'].dt.strftime('%B')
df['quarter'] = 'Q' + df['order_date'].dt.quarter.astype(str)
df['year_month'] = df['order_date'].dt.strftime('%Y-%m')
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days
df['profit_margin'] = (df['profit'] / df['sales']).round(4)

df[['order_date', 'ship_date', 'year', 'month', 'year_month', 'shipping_days', 'sales', 'profit', 'profit_margin']].head(5)

## 3. Post-Cleaning Validation Checks
Confirm data integrity, ranges, and domain constraints.

In [ ]:
print(f'Cleaned dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Order Date range: {df["order_date"].min().date()} to {df["order_date"].max().date()}')
print(f'Ship Date range:  {df["ship_date"].min().date()} to {df["ship_date"].max().date()}')
print(f'Shipping Days range: {df["shipping_days"].min()} to {df["shipping_days"].max()} days')
print(f'Sales range: ${df["sales"].min():.2f} to ${df["sales"].max():,.2f}')
print(f'Discount range: {df["discount"].min()} to {df["discount"].max()}')
print(f'Profit range: ${df["profit"].min():,.2f} to ${df["profit"].max():,.2f}')
print(f'Loss-making transactions preserved: {(df["profit"] < 0).sum():,} rows')

# Export cleaned file
df.to_csv(CLEANED_DATA_PATH, index=False)
print(f'Successfully saved cleaned dataset to {CLEANED_DATA_PATH}')